Section 1 — Data Ingestion & Validation

This section has the following chunks:

1. Install & Import Libraries — install pandasql and import all required libraries including Pandas, NumPy, Plotly, and Seaborn
2. Load the CSVs — load all 9 Olist dataset files into individual Pandas DataFrames and confirm they loaded correctly with a shape summary
3. Explore the Structure — inspect column names, data types, and sample rows for each table to understand the dataset before any processing
4. Automated Data Quality Checks — run automated checks for missing values, duplicate rows, and incorrectly typed columns across all tables and generate a quality report
5. Referential Integrity Checks — verify that foreign keys match across related tables (e.g. every order_id in order_items exists in orders) to catch orphaned records before joining

In [21]:
# ─────────────────────────────────────────────
# SECTION 1: Data Ingestion & Validation
# ─────────────────────────────────────────────

# We start by installing any libraries that aren't available in Colab by default.
# pandasql lets us run SQL queries directly on Pandas DataFrames —
# this is important because the JD calls for SQL fluency, and we want to
# demonstrate that throughout the project.

!pip install pandasql --quiet

# ── Standard Imports ──
import pandas as pd          # core data manipulation
import numpy as np           # numerical operations
import pandasql as psql      # run SQL on DataFrames
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os                    # for file path handling
import warnings
warnings.filterwarnings('ignore')

# ── Display settings ──
pd.set_option('display.max_columns', None)   # show all columns when previewing
pd.set_option('display.float_format', '{:.2f}'.format)  # clean decimal display

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [22]:
# ── Load all 9 CSV files into Pandas DataFrames ──
# Each file represents a different table in the Olist e-commerce database.
# We load them individually so we can inspect, validate, and join them
# in a controlled way — mimicking how you'd work with tables in a real database.

orders = pd.read_csv('/olist_orders_dataset.csv')
customers = pd.read_csv('/olist_customers_dataset.csv')
order_items = pd.read_csv('/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/olist_order_reviews_dataset.csv')
products = pd.read_csv('/olist_products_dataset.csv')
sellers = pd.read_csv('/olist_sellers_dataset.csv')
geolocation = pd.read_csv('/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('/product_category_name_translation.csv')

# ── Store all DataFrames in a dictionary for easy iteration later ──
# This is useful when we want to run the same checks across all tables
# without repeating code (e.g., null checks, shape checks).

all_dfs = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'category_translation': category_translation
}

# ── Quick confirmation that all files loaded correctly ──
# We print the shape (rows x columns) of each table so we can immediately
# spot if anything loaded empty or incorrectly.

print("All datasets loaded successfully\n")
print(f"{'Dataset':<25} {'Rows':>10} {'Columns':>10}")
print("-" * 47)
for name, df in all_dfs.items():
    print(f"{name:<25} {df.shape[0]:>10,} {df.shape[1]:>10}")

All datasets loaded successfully

Dataset                         Rows    Columns
-----------------------------------------------
orders                        99,441          8
customers                     99,441          5
order_items                  112,650          7
order_payments               103,886          5
order_reviews                 99,224          7
products                      32,951          9
sellers                        3,095          4
geolocation                1,000,163          5
category_translation              71          2


In [23]:
# ── Explore the structure of each dataset ──
# Before doing any analysis or transformation, we need to understand what
# each table looks like — column names, data types, and a sample of the data.
# This is the first step any data analyst does when working with a new dataset.
# It helps us catch obvious issues early (wrong types, unexpected columns, etc.)

def explore_dataframe(name, df):
    print(f"\n{'='*60}")
    print(f"📋 TABLE: {name.upper()}")
    print(f"{'='*60}")

    # Shape tells us the size of the table
    print(f"\n🔢 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    # Column names and their data types — important for knowing what
    # needs to be converted (e.g., date columns stored as strings)
    print(f"\n📌 Columns & Data Types:")
    print(df.dtypes.to_string())

    # A sample of the first 3 rows so we can see actual values
    print(f"\n👀 Sample Data (first 3 rows):")
    display(df.head(3))

# ── Run exploration on all tables ──
# We skip geolocation here because it has 1M+ rows and is mostly
# zip code coordinates — not the focus of our analysis
for name, df in all_dfs.items():
    if name != 'geolocation':
        explore_dataframe(name, df)


📋 TABLE: ORDERS

🔢 Shape: 99,441 rows × 8 columns

📌 Columns & Data Types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object

👀 Sample Data (first 3 rows):


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



📋 TABLE: CUSTOMERS

🔢 Shape: 99,441 rows × 5 columns

📌 Columns & Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object

👀 Sample Data (first 3 rows):


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



📋 TABLE: ORDER_ITEMS

🔢 Shape: 112,650 rows × 7 columns

📌 Columns & Data Types:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64

👀 Sample Data (first 3 rows):


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00



📋 TABLE: SELLERS

🔢 Shape: 3,095 rows × 4 columns

📌 Columns & Data Types:
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object

👀 Sample Data (first 3 rows):


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



📋 TABLE: CATEGORY_TRANSLATION

🔢 Shape: 71 rows × 2 columns

📌 Columns & Data Types:
product_category_name            object
product_category_name_english    object

👀 Sample Data (first 3 rows):


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [24]:
# ── Automated Data Quality Checks ──
# In a real enterprise data pipeline, you'd have automated checks that run
# every time new data comes in. Here we simulate that by building a reusable
# function that checks for the most common data quality issues:
# 1. Missing values — columns with nulls that could break downstream analysis
# 2. Duplicate rows — exact duplicates that would skew aggregations
# 3. Data type issues — columns that should be numeric or datetime but aren't

def data_quality_check(name, df):
    print(f"\n{'='*60}")
    print(f"DATA QUALITY REPORT: {name.upper()}")
    print(f"{'='*60}")

    # ── 1. Missing Values ──
    # We calculate both the count and percentage of nulls per column.
    # Percentage is more meaningful than count alone because a dataset
    # with 100k rows and 500 nulls in one column is very different from
    # one with 1000 rows and 500 nulls.
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    null_report = pd.DataFrame({
        'Missing Count': null_counts,
        'Missing %': null_pct
    })
    null_report = null_report[null_report['Missing Count'] > 0]

    if null_report.empty:
        print("\nNo missing values found")
    else:
        print(f"\nMissing Values:")
        print(null_report.to_string())

    # ── 2. Duplicate Rows ──
    # Duplicate rows can happen due to pipeline errors, double ingestion,
    # or bugs in upstream systems. They silently inflate metrics like
    # revenue or order counts if not caught early.
    dup_count = df.duplicated().sum()
    if dup_count == 0:
        print(f"\nNo duplicate rows found")
    else:
        print(f"\nDuplicate Rows: {dup_count:,} ({dup_count/len(df)*100:.2f}%)")

    # ── 3. Data Type Issues ──
    # We flag any column whose name suggests it should be a date or number
    # but is currently stored as a string (object). These need to be
    # converted before we can do time-series analysis or arithmetic on them.
    object_cols = df.select_dtypes(include='object').columns.tolist()
    suspected_dates = [c for c in object_cols if 'date' in c.lower() or 'timestamp' in c.lower()]
    suspected_nums = [c for c in object_cols if any(x in c.lower() for x in ['price', 'value', 'freight', 'payment'])]

    if suspected_dates:
        print(f"\nColumns that should be datetime: {suspected_dates}")
    if suspected_nums:
        print(f"\nColumns that should be numeric: {suspected_nums}")
    if not suspected_dates and not suspected_nums:
        print(f"\nNo obvious data type issues detected")

# ── Run quality checks on all tables except geolocation ──
for name, df in all_dfs.items():
    if name != 'geolocation':
        data_quality_check(name, df)

# ── Summary ──
print(f"\n{'='*60}")
print("OVERALL DATASET SUMMARY")
print(f"{'='*60}")
total_rows = sum(df.shape[0] for name, df in all_dfs.items() if name != 'geolocation')
total_cols = sum(df.shape[1] for name, df in all_dfs.items() if name != 'geolocation')
print(f"Total rows across all tables : {total_rows:,}")
print(f"Total columns across all tables: {total_cols}")
print(f"\nData Quality Check Complete")


DATA QUALITY REPORT: ORDERS

Missing Values:
                               Missing Count  Missing %
order_approved_at                        160       0.16
order_delivered_carrier_date            1783       1.79
order_delivered_customer_date           2965       2.98

No duplicate rows found

Columns that should be datetime: ['order_purchase_timestamp', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

DATA QUALITY REPORT: CUSTOMERS

No missing values found

No duplicate rows found

No obvious data type issues detected

DATA QUALITY REPORT: ORDER_ITEMS

No missing values found

No duplicate rows found

Columns that should be datetime: ['shipping_limit_date']

DATA QUALITY REPORT: ORDER_PAYMENTS

No missing values found

No duplicate rows found

Columns that should be numeric: ['payment_type']

DATA QUALITY REPORT: ORDER_REVIEWS

Missing Values:
                        Missing Count  Missing %
review_comment_title            87656      

In [25]:
# ── Referential Integrity Checks ──
# Referential integrity means that keys that are supposed to match across
# tables actually do. For example, every order_id in the order_items table
# should exist in the orders table. If they don't, it means we have orphaned
# records that will cause silent data loss when we join tables later.
# This is a critical check in any enterprise data governance effort.

def referential_integrity_check(child_name, child_df, child_key,
                                 parent_name, parent_df, parent_key):

    child_keys = set(child_df[child_key].dropna())
    parent_keys = set(parent_df[parent_key].dropna())

    # Records in child table whose key does not exist in the parent table
    orphaned = child_keys - parent_keys
    orphan_pct = (len(orphaned) / len(child_keys) * 100) if child_keys else 0

    status = "PASS" if len(orphaned) == 0 else "FAIL"
    print(f"  [{status}] {child_name}.{child_key} -> {parent_name}.{parent_key} "
          f"| Orphaned keys: {len(orphaned):,} ({orphan_pct:.2f}%)")

print("=" * 60)
print("REFERENTIAL INTEGRITY REPORT")
print("=" * 60)

print("\n-- order_items --")
# Every item should belong to a valid order
referential_integrity_check('order_items', order_items, 'order_id',
                             'orders', orders, 'order_id')
# Every item should reference a valid product
referential_integrity_check('order_items', order_items, 'product_id',
                             'products', products, 'product_id')
# Every item should reference a valid seller
referential_integrity_check('order_items', order_items, 'seller_id',
                             'sellers', sellers, 'seller_id')

print("\n-- order_payments --")
# Every payment should belong to a valid order
referential_integrity_check('order_payments', order_payments, 'order_id',
                             'orders', orders, 'order_id')

print("\n-- order_reviews --")
# Every review should belong to a valid order
referential_integrity_check('order_reviews', order_reviews, 'order_id',
                             'orders', orders, 'order_id')

print("\n-- customers --")
# Every order should belong to a valid customer
referential_integrity_check('orders', orders, 'customer_id',
                             'customers', customers, 'customer_id')

print("\n-- sellers --")
# Every seller should have a valid zip code in geolocation
referential_integrity_check('sellers', sellers, 'seller_zip_code_prefix',
                             'geolocation', geolocation, 'geolocation_zip_code_prefix')

print("\n" + "=" * 60)
print("Referential Integrity Check Complete")
print("=" * 60)

REFERENTIAL INTEGRITY REPORT

-- order_items --
  [PASS] order_items.order_id -> orders.order_id | Orphaned keys: 0 (0.00%)
  [PASS] order_items.product_id -> products.product_id | Orphaned keys: 0 (0.00%)
  [PASS] order_items.seller_id -> sellers.seller_id | Orphaned keys: 0 (0.00%)

-- order_payments --
  [PASS] order_payments.order_id -> orders.order_id | Orphaned keys: 0 (0.00%)

-- order_reviews --
  [PASS] order_reviews.order_id -> orders.order_id | Orphaned keys: 0 (0.00%)

-- customers --
  [PASS] orders.customer_id -> customers.customer_id | Orphaned keys: 0 (0.00%)

-- sellers --
  [FAIL] sellers.seller_zip_code_prefix -> geolocation.geolocation_zip_code_prefix | Orphaned keys: 7 (0.31%)

Referential Integrity Check Complete


Section 2 — Data Transformation & Feature Engineering

This section has the following chunks:

1. Fix Data Types — convert date columns from string to datetime, fix numeric columns

2. Clean the Data — handle nulls, drop duplicates, standardize text

3. Join the Tables with SQL — build a unified analytical dataset using SQL queries on the DataFrames

4. Feature Engineering — create new columns like delivery delay, revenue per order, seller performance score

5. PySpark Transformation — repeat one key transformation using PySpark to demonstrate big data fluency

In [26]:
# ── SECTION 2: Data Transformation & Feature Engineering ──

# ── Fix Data Types ──
# From our data quality checks in Section 1, we identified several columns
# that are stored as strings but should be datetime or numeric types.
# If we don't fix these now, any time-based analysis (trends, delays,
# forecasting) and arithmetic operations (revenue, totals) will fail or
# produce incorrect results.

# ── Convert date/timestamp columns to datetime in the orders table ──
# These columns represent key timestamps in the order lifecycle.
# We use errors='coerce' which means if a value can't be parsed as a date,
# it becomes NaT (null) instead of crashing the entire conversion.

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("Orders date columns converted:")
print(orders[date_columns].dtypes)

# ── Convert date columns in order_reviews ──
# Review creation and answer timestamps are also stored as strings
review_date_cols = ['review_creation_date', 'review_answer_timestamp']
for col in review_date_cols:
    order_reviews[col] = pd.to_datetime(order_reviews[col], errors='coerce')

print("\nOrder reviews date columns converted:")
print(order_reviews[review_date_cols].dtypes)

# ── Convert date column in order_items ──
order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date'], errors='coerce'
)
print("\nOrder items date column converted:")
print(order_items[['shipping_limit_date']].dtypes)

# ── Ensure numeric columns are correctly typed ──
# price and freight_value should already be float but we enforce it
# to be safe, especially if any values were read as strings
order_items['price'] = pd.to_numeric(order_items['price'], errors='coerce')
order_items['freight_value'] = pd.to_numeric(order_items['freight_value'], errors='coerce')
order_payments['payment_value'] = pd.to_numeric(order_payments['payment_value'], errors='coerce')

print("\nNumeric columns confirmed:")
print(order_items[['price', 'freight_value']].dtypes)
print(order_payments[['payment_value']].dtypes)

print("\nData type fixes complete")

Orders date columns converted:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Order reviews date columns converted:
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

Order items date column converted:
shipping_limit_date    datetime64[ns]
dtype: object

Numeric columns confirmed:
price            float64
freight_value    float64
dtype: object
payment_value    float64
dtype: object

Data type fixes complete


In [27]:
# ── Clean the Data ──
# Now that data types are fixed, we handle the actual data quality issues
# flagged in Section 1. The goal here is to produce a clean version of
# each table that we can safely join and analyze without risk of nulls
# or duplicates corrupting our metrics.

# ── 1. Remove duplicate rows ──
# We drop exact duplicate rows across all key tables. Duplicates can
# inflate metrics like revenue or order counts, leading to wrong KPIs.

before_orders = len(orders)
before_items = len(order_items)
before_payments = len(order_payments)
before_reviews = len(order_reviews)

orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
order_payments = order_payments.drop_duplicates()
order_reviews = order_reviews.drop_duplicates()

print("Duplicates removed:")
print(f"  orders        : {before_orders - len(orders):,} rows removed")
print(f"  order_items   : {before_items - len(order_items):,} rows removed")
print(f"  order_payments: {before_payments - len(order_payments):,} rows removed")
print(f"  order_reviews : {before_reviews - len(order_reviews):,} rows removed")

# ── 2. Handle missing values in orders ──
# order_approved_at, order_delivered_carrier_date, and
# order_delivered_customer_date can be null for cancelled or
# in-progress orders. We keep these rows but will exclude them
# from delivery-related KPIs later so they don't skew results.
# We only drop rows where the core identifier (order_id) is null
# since those records are completely unusable.

orders = orders.dropna(subset=['order_id', 'customer_id'])
print(f"\nOrders after dropping rows with null order_id/customer_id: {len(orders):,}")

# ── 3. Handle missing values in products ──
# Some products are missing category names. We fill these with
# 'unknown' rather than dropping the rows, because the product
# may still have valid sales data we want to keep.

products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Product dimension columns (weight, size) have some nulls.
# We fill with the median of each column — a safe choice that is
# robust to outliers compared to using the mean.
dimension_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
for col in dimension_cols:
    products[col] = products[col].fillna(products[col].median())

print(f"\nProducts null values after cleaning:")
print(products.isnull().sum())

# ── 4. Handle missing values in order_reviews ──
# Review comments are optional — many customers leave a score
# without writing a comment. We fill nulls with empty string
# so text-based operations don't break later.
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].fillna('')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('')

# ── 5. Standardize text columns ──
# We lowercase and strip whitespace from category names and state codes
# to avoid mismatches during groupby operations (e.g., 'SP' vs 'sp')
products['product_category_name'] = (
    products['product_category_name'].str.lower().str.strip()
)
customers['customer_state'] = (
    customers['customer_state'].str.upper().str.strip()
)
sellers['seller_state'] = (
    sellers['seller_state'].str.upper().str.strip()
)

print("\nText standardization complete")
print("\nData cleaning complete")

Duplicates removed:
  orders        : 0 rows removed
  order_items   : 0 rows removed
  order_payments: 0 rows removed
  order_reviews : 0 rows removed

Orders after dropping rows with null order_id/customer_id: 99,441

Products null values after cleaning:
product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64

Text standardization complete

Data cleaning complete


In [28]:
# ── Join the Tables with SQL ──
# We use pandasql for most joins, but replace the correlated subquery
# (dominant category per order) with a Pandas operation — pandasql is
# too slow for that pattern at 100k+ rows. Everything else stays as SQL.

def sql(query):
    return psql.sqldf(query, locals() | globals())

# ── Step 1: Translate product category names to English ──
products_translated = sql("""
    SELECT
        p.product_id,
        p.product_category_name,
        COALESCE(t.product_category_name_english, 'unknown') AS category_english,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm
    FROM products p
    LEFT JOIN category_translation t
        ON p.product_category_name = t.product_category_name
""")
print(f"Products with English categories: {len(products_translated):,} rows")

# ── Step 2: Aggregate payments per order ──
order_payments_agg = sql("""
    SELECT
        order_id,
        SUM(payment_value)         AS total_payment_value,
        COUNT(*)                   AS payment_installments,
        GROUP_CONCAT(payment_type) AS payment_types
    FROM order_payments
    GROUP BY order_id
""")
print(f"Aggregated payments: {len(order_payments_agg):,} rows")

# ── Step 3: Aggregate order items per order ──
order_items_agg = sql("""
    SELECT
        order_id,
        COUNT(*)                   AS item_count,
        SUM(price)                 AS total_item_price,
        SUM(freight_value)         AS total_freight_value,
        SUM(price + freight_value) AS total_order_value
    FROM order_items
    GROUP BY order_id
""")
print(f"Aggregated order items: {len(order_items_agg):,} rows")

# ── Step 4: Get dominant product category per order using Pandas ──
# We avoid a correlated subquery here because pandasql is extremely slow
# for that pattern on large datasets. Instead we use Pandas groupby + idxmax
# which does the same thing — find the highest revenue category per order —
# but runs in seconds rather than minutes.

# First merge order_items with translated product categories
items_with_category = order_items.merge(
    products_translated[['product_id', 'category_english']],
    on='product_id',
    how='left'
)

# Sum revenue by order and category, then pick the top category per order
category_revenue = (
    items_with_category
    .groupby(['order_id', 'category_english'])['price']
    .sum()
    .reset_index()
)
# idxmax gives us the index of the highest revenue category per order
idx = category_revenue.groupby('order_id')['price'].idxmax()
order_category = category_revenue.loc[idx, ['order_id', 'category_english']].reset_index(drop=True)

print(f"Order categories: {len(order_category):,} rows")

# ── Step 5: Aggregate reviews per order ──
order_reviews_agg = sql("""
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score,
        COUNT(*)          AS review_count
    FROM order_reviews
    GROUP BY order_id
""")
print(f"Aggregated reviews: {len(order_reviews_agg):,} rows")

# ── Step 6: Build the final unified analytical dataset ──
# We join everything into one master table using SQL.
# LEFT JOINs ensure we keep all orders even if some have missing
# payments, reviews, or items — we never want to silently drop orders.
master_df = sql("""
    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        c.customer_state,
        c.customer_city,
        oi.item_count,
        oi.total_item_price,
        oi.total_freight_value,
        oi.total_order_value,
        op.total_payment_value,
        op.payment_types,
        r.avg_review_score,
        r.review_count,
        oc.category_english AS primary_category
    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id
    LEFT JOIN order_items_agg oi
        ON o.order_id = oi.order_id
    LEFT JOIN order_payments_agg op
        ON o.order_id = op.order_id
    LEFT JOIN order_reviews_agg r
        ON o.order_id = r.order_id
    LEFT JOIN order_category oc
        ON o.order_id = oc.order_id
""")

print(f"\nMaster dataset: {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")
display(master_df.head(3))
print("\nSQL joins complete — master dataset ready")

Products with English categories: 32,951 rows
Aggregated payments: 99,440 rows
Aggregated order items: 98,666 rows
Order categories: 98,666 rows
Aggregated reviews: 98,673 rows

Master dataset: 99,441 rows x 19 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,item_count,total_item_price,total_freight_value,total_order_value,total_payment_value,payment_types,avg_review_score,review_count,primary_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33.000000,2017-10-02 11:07:15.000000,2017-10-04 19:55:00.000000,2017-10-10 21:25:13.000000,2017-10-18 00:00:00.000000,SP,sao paulo,1.00,29.99,8.72,38.71,38.71,"credit_card,voucher,voucher",4.00,1.00,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37.000000,2018-07-26 03:24:27.000000,2018-07-26 14:31:00.000000,2018-08-07 15:27:45.000000,2018-08-13 00:00:00.000000,BA,barreiras,1.00,118.70,22.76,141.46,141.46,boleto,4.00,1.00,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49.000000,2018-08-08 08:55:23.000000,2018-08-08 13:50:00.000000,2018-08-17 18:06:29.000000,2018-09-04 00:00:00.000000,GO,vianopolis,1.00,159.90,19.22,179.12,179.12,credit_card,5.00,1.00,auto



SQL joins complete — master dataset ready


In [29]:
# ── Feature Engineering ──
# Raw columns alone are not enough for meaningful KPI analysis.
# We create new derived columns (features) that directly represent
# business concepts like delivery delay, revenue, and order speed.
# These features are what KPI dashboards and anomaly detection models
# actually operate on — not the raw timestamps or prices.

# ── 1. Convert date columns in master_df to datetime ──
# The SQL join returns all columns as strings, so we need to re-convert
# our date columns before we can do any time-based calculations.

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col], errors='coerce')

# ── 2. Delivery delay (in days) ──
# Delivery delay = actual delivery date minus estimated delivery date.
# Positive value means the order arrived late, negative means early.
# This is one of the most important operational KPIs for e-commerce.
master_df['delivery_delay_days'] = (
    master_df['order_delivered_customer_date'] -
    master_df['order_estimated_delivery_date']
).dt.days

# ── 3. Actual delivery time (in days) ──
# How long it actually took from purchase to delivery.
# Useful for understanding fulfillment speed independent of the estimate.
master_df['actual_delivery_days'] = (
    master_df['order_delivered_customer_date'] -
    master_df['order_purchase_timestamp']
).dt.days

# ── 4. Approval time (in days) ──
# How long it took for the order to be approved after purchase.
# Slow approval times can indicate payment or fraud review bottlenecks.
master_df['approval_time_days'] = (
    master_df['order_approved_at'] -
    master_df['order_purchase_timestamp']
).dt.days

# ── 5. On-time delivery flag ──
# A binary flag: 1 if the order was delivered on or before the estimated
# date, 0 if it was late. This feeds directly into our on-time delivery
# rate KPI in Section 3.
master_df['is_on_time'] = (
    master_df['delivery_delay_days'] <= 0
).astype(int)

# ── 6. Late delivery flag ──
# Explicit flag for late orders — useful for filtering and anomaly detection.
master_df['is_late'] = (
    master_df['delivery_delay_days'] > 0
).astype(int)

# ── 7. Revenue features ──
# total_order_value already exists from our aggregation but we add
# freight as a percentage of order value — high freight ratios indicate
# inefficient logistics or low average order values.
master_df['freight_pct'] = (
    master_df['total_freight_value'] /
    master_df['total_order_value'].replace(0, np.nan) * 100
).round(2)

# ── 8. Purchase date parts ──
# Breaking out year, month, day of week from the purchase timestamp
# allows us to analyze seasonality and weekly patterns in Section 3.
master_df['purchase_year'] = master_df['order_purchase_timestamp'].dt.year
master_df['purchase_month'] = master_df['order_purchase_timestamp'].dt.month
master_df['purchase_month_name'] = master_df['order_purchase_timestamp'].dt.strftime('%b')
master_df['purchase_dayofweek'] = master_df['order_purchase_timestamp'].dt.day_name()
master_df['purchase_date'] = master_df['order_purchase_timestamp'].dt.date

# ── 9. Review sentiment bucketing ──
# We bucket review scores into sentiment categories so we can analyze
# customer satisfaction at a higher level than individual star ratings.
def sentiment_bucket(score):
    if pd.isna(score):
        return 'no review'
    elif score >= 4:
        return 'positive'
    elif score == 3:
        return 'neutral'
    else:
        return 'negative'

master_df['review_sentiment'] = master_df['avg_review_score'].apply(sentiment_bucket)

# ── Summary of new features ──
new_features = [
    'delivery_delay_days', 'actual_delivery_days', 'approval_time_days',
    'is_on_time', 'is_late', 'freight_pct',
    'purchase_year', 'purchase_month', 'purchase_dayofweek',
    'purchase_date', 'review_sentiment'
]

print("Feature engineering complete")
print(f"\nNew features added: {len(new_features)}")
print(f"\nMaster dataset shape: {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")
print("\nSample of new features:")
display(master_df[new_features].head(5))

print("\nFeature value summary:")
print(master_df[['delivery_delay_days', 'actual_delivery_days',
                  'approval_time_days', 'freight_pct']].describe().round(2))

Feature engineering complete

New features added: 11

Master dataset shape: 99,441 rows x 31 columns

Sample of new features:


,delivery_delay_days,actual_delivery_days,approval_time_days,is_on_time,is_late,freight_pct,purchase_year,purchase_month,purchase_dayofweek,purchase_date,review_sentiment
0,-8.00,8.00,0.00,1,0,22.53,2017,10,Monday,2017-10-02,positive
1,-6.00,13.00,1.00,1,0,16.09,2018,7,Tuesday,2018-07-24,positive
2,-18.00,9.00,0.00,1,0,10.73,2018,8,Wednesday,2018-08-08,positive
3,-13.00,13.00,0.00,1,0,37.67,2017,11,Saturday,2017-11-18,positive
4,-10.00,2.00,0.00,1,0,30.47,2018,2,Tuesday,2018-02-13,positive



Feature value summary:
       delivery_delay_days  actual_delivery_days  approval_time_days  \
count             96476.00              96476.00            99281.00   
mean                -11.88                 12.09                0.27   
std                  10.18                  9.55                0.99   
min                -147.00                  0.00                0.00   
25%                 -17.00                  6.00                0.00   
50%                 -12.00                 10.00                0.00   
75%                  -7.00                 15.00                0.00   
max                 188.00                209.00              187.00   

       freight_pct  
count     98666.00  
mean         20.88  
std          12.57  
min           0.00  
25%          11.65  
50%          18.33  
75%          27.55  
max          95.55  


In [30]:
# ── PySpark Transformation ──
# In production enterprise environments, datasets are often too large
# for single-machine Pandas processing. PySpark allows the same
# transformations to run across a distributed cluster (e.g., AWS EMR).
# Here we demonstrate the same delivery delay aggregation we did in
# Pandas, but using PySpark — showing we can operate in both environments.
# This directly addresses the PySpark requirement in the job description.

# ── Install and set up PySpark in Colab ──
!pip install pyspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize a local Spark session
# In production this would point to an EMR or Databricks cluster
spark = SparkSession.builder \
    .appName("OlistAnalytics") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("Spark session started successfully")

# ── Convert our master Pandas DataFrame to a Spark DataFrame ──
# In a real pipeline, Spark would read directly from S3, HDFS, or a
# data lake. Here we convert from Pandas to simulate that workflow
# without needing external storage infrastructure.
spark_df = spark.createDataFrame(master_df[[
    'order_id',
    'order_status',
    'customer_state',
    'total_order_value',
    'total_freight_value',
    'delivery_delay_days',
    'actual_delivery_days',
    'is_on_time',
    'is_late',
    'primary_category',
    'avg_review_score',
    'purchase_year',
    'purchase_month'
]].astype(str).fillna('null'))

print(f"\nSpark DataFrame created: {spark_df.count():,} rows")
spark_df.printSchema()

# ── PySpark Transformation 1: Delivery performance by state ──
# We aggregate delivery metrics at the customer state level.
# This tells us which regions have the worst delivery performance —
# a key operational insight for logistics planning.
delivery_by_state = spark_df \
    .filter(F.col('delivery_delay_days') != 'null') \
    .withColumn('delivery_delay_days', F.col('delivery_delay_days').cast('float')) \
    .withColumn('is_late', F.col('is_late').cast('float')) \
    .withColumn('total_order_value', F.col('total_order_value').cast('float')) \
    .groupBy('customer_state') \
    .agg(
        F.count('order_id').alias('total_orders'),
        F.round(F.avg('delivery_delay_days'), 2).alias('avg_delay_days'),
        F.round(F.avg('is_late') * 100, 2).alias('late_delivery_pct'),
        F.round(F.sum('total_order_value'), 2).alias('total_revenue')
    ) \
    .orderBy(F.desc('late_delivery_pct'))

print("\nDelivery Performance by State (PySpark):")
delivery_by_state.show(10)

# ── PySpark Transformation 2: Revenue and order volume by category ──
# We compute revenue and average order value per product category.
# This identifies which categories drive the most business value.
revenue_by_category = spark_df \
    .filter(F.col('primary_category') != 'null') \
    .withColumn('total_order_value', F.col('total_order_value').cast('float')) \
    .withColumn('avg_review_score', F.col('avg_review_score').cast('float')) \
    .groupBy('primary_category') \
    .agg(
        F.count('order_id').alias('total_orders'),
        F.round(F.sum('total_order_value'), 2).alias('total_revenue'),
        F.round(F.avg('total_order_value'), 2).alias('avg_order_value'),
        F.round(F.avg('avg_review_score'), 2).alias('avg_review_score')
    ) \
    .orderBy(F.desc('total_revenue'))

print("\nRevenue by Product Category (PySpark):")
revenue_by_category.show(10)

# ── Convert PySpark results back to Pandas for visualization later ──
# Spark is used for the heavy transformation; Pandas and Plotly handle
# the visualization layer — this is a common real-world pattern.
delivery_by_state_pd = delivery_by_state.toPandas()
revenue_by_category_pd = revenue_by_category.toPandas()

print("\nPySpark transformations complete")
print("Results converted back to Pandas for visualization")
print("\nSection 2 complete — master dataset and features ready for KPI analysis")

Spark version: 4.0.2
Spark session started successfully

Spark DataFrame created: 99,441 rows
root
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_order_value: string (nullable = true)
 |-- total_freight_value: string (nullable = true)
 |-- delivery_delay_days: string (nullable = true)
 |-- actual_delivery_days: string (nullable = true)
 |-- is_on_time: string (nullable = true)
 |-- is_late: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- avg_review_score: string (nullable = true)
 |-- purchase_year: string (nullable = true)
 |-- purchase_month: string (nullable = true)


Delivery Performance by State (PySpark):
+--------------+------------+--------------+-----------------+-------------+
|customer_state|total_orders|avg_delay_days|late_delivery_pct|total_revenue|
+--------------+------------+--------------+-----------------+-------------+
|            AL|         4

Section 3 — KPI Framework & EDA

This section has the following chunks:

1. Finance KPIs — total revenue, average order value, revenue trends over time, revenue by category

2. Operations KPIs — on-time delivery rate, average delivery delay, delay distribution, delivery performance by state

3. Seller & Customer KPIs — review score distribution, sentiment breakdown, day of week and monthly order patterns

4. KPI Summary Dashboard — one consolidated view of all key metrics

In [31]:
# ── SECTION 3: KPI Framework & EDA ──

# ── Finance KPIs ──
# Finance KPIs answer the most fundamental business question: how much
# money are we making, where is it coming from, and is it growing?
# We define and compute four core finance KPIs here:
# 1. Total Revenue
# 2. Average Order Value (AOV)
# 3. Monthly Revenue Trend
# 4. Revenue by Product Category

# ── Filter to delivered orders only for financial metrics ──
# We only count revenue from orders that were actually delivered.
# Cancelled or unavailable orders should not be included in revenue
# figures as the money was not collected.
delivered = master_df[master_df['order_status'] == 'delivered'].copy()
print(f"Delivered orders: {len(delivered):,} out of {len(master_df):,} total orders")

# ────────────────────────────────────────────
# KPI 1: Total Revenue
# ────────────────────────────────────────────
total_revenue = delivered['total_order_value'].sum()
print(f"\nKPI 1 - Total Revenue: R$ {total_revenue:,.2f}")

# ────────────────────────────────────────────
# KPI 2: Average Order Value (AOV)
# ────────────────────────────────────────────
# AOV tells us how much a typical customer spends per order.
# A rising AOV means customers are buying more per transaction,
# which is more efficient than acquiring new customers.
aov = delivered['total_order_value'].mean()
print(f"KPI 2 - Average Order Value (AOV): R$ {aov:,.2f}")

# ────────────────────────────────────────────
# KPI 3: Monthly Revenue Trend
# ────────────────────────────────────────────
# We aggregate revenue by year-month to see how the business
# grew over time. This is the most important trend chart for
# any finance or operations review.
monthly_revenue = (
    delivered
    .groupby(['purchase_year', 'purchase_month'])['total_order_value']
    .sum()
    .reset_index()
)

# Create a proper date column for clean x-axis plotting
monthly_revenue['period'] = pd.to_datetime(
    monthly_revenue['purchase_year'].astype(str) + '-' +
    monthly_revenue['purchase_month'].astype(str).str.zfill(2)
)
monthly_revenue = monthly_revenue.sort_values('period')

# Remove the last month if incomplete — partial months skew trend lines
# downward and create a false impression of declining revenue
last_month = monthly_revenue['period'].max()
monthly_revenue = monthly_revenue[monthly_revenue['period'] < last_month]

fig1 = px.line(
    monthly_revenue,
    x='period',
    y='total_order_value',
    title='Monthly Revenue Trend (Delivered Orders)',
    labels={'period': 'Month', 'total_order_value': 'Revenue (R$)'},
    markers=True
)
fig1.update_layout(
    title_font_size=18,
    xaxis_title='Month',
    yaxis_title='Revenue (R$)',
    hovermode='x unified',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey')
)
fig1.update_traces(line_color='#2563eb', marker_color='#2563eb')
fig1.show()

# ── Insight ──
print("\nInsight - Monthly Revenue Trend:")
peak_month = monthly_revenue.loc[monthly_revenue['total_order_value'].idxmax()]
print(f"  Peak revenue month: {peak_month['period'].strftime('%B %Y')} "
      f"(R$ {peak_month['total_order_value']:,.2f})")
avg_monthly = monthly_revenue['total_order_value'].mean()
print(f"  Average monthly revenue: R$ {avg_monthly:,.2f}")

# ────────────────────────────────────────────
# KPI 4: Revenue by Product Category
# ────────────────────────────────────────────
# Understanding which categories drive revenue helps prioritize
# inventory, marketing spend, and supplier relationships.
category_revenue = (
    delivered
    .groupby('primary_category')['total_order_value']
    .sum()
    .reset_index()
    .sort_values('total_order_value', ascending=False)
    .head(15)  # top 15 categories for readability
)

fig2 = px.bar(
    category_revenue,
    x='total_order_value',
    y='primary_category',
    orientation='h',
    title='Top 15 Product Categories by Revenue',
    labels={'total_order_value': 'Total Revenue (R$)', 'primary_category': 'Category'},
    color='total_order_value',
    color_continuous_scale='Blues'
)
fig2.update_layout(
    title_font_size=18,
    plot_bgcolor='white',
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    xaxis=dict(gridcolor='lightgrey')
)
fig2.show()

# ── Insight ──
print("\nInsight - Revenue by Category:")
top_cat = category_revenue.iloc[0]
top5_pct = (
    category_revenue.head(5)['total_order_value'].sum() /
    delivered['total_order_value'].sum() * 100
)
print(f"  Top category: {top_cat['primary_category']} "
      f"(R$ {top_cat['total_order_value']:,.2f})")
print(f"  Top 5 categories account for {top5_pct:.1f}% of total revenue")

print("\nFinance KPIs complete")

Delivered orders: 96,478 out of 99,441 total orders

KPI 1 - Total Revenue: R$ 15,419,773.75
KPI 2 - Average Order Value (AOV): R$ 159.83



Insight - Monthly Revenue Trend:
  Peak revenue month: November 2017 (R$ 1,153,364.20)
  Average monthly revenue: R$ 656,103.73



Insight - Revenue by Category:
  Top category: health_beauty (R$ 1,413,225.63)
  Top 5 categories account for 39.3% of total revenue

Finance KPIs complete


In [32]:
# ── Operations KPIs ──
# Operations KPIs measure how well the business fulfills orders.
# Poor delivery performance directly impacts customer satisfaction
# and repeat purchase rates. We define four core operations KPIs:
# 1. On-Time Delivery Rate
# 2. Average Delivery Delay
# 3. Delivery Delay Distribution
# 4. Delivery Performance by State

# ── Filter to orders that have actual delivery dates ──
# We can only measure delivery performance on orders where we know
# both the actual and estimated delivery dates.
delivery_df = delivered.dropna(subset=[
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'delivery_delay_days'
]).copy()

print(f"Orders with delivery data: {len(delivery_df):,}")

# ────────────────────────────────────────────
# KPI 5: On-Time Delivery Rate
# ────────────────────────────────────────────
# The percentage of orders delivered on or before the estimated date.
# This is the single most important fulfillment KPI for e-commerce.
on_time_rate = delivery_df['is_on_time'].mean() * 100
late_rate = 100 - on_time_rate

print(f"\nKPI 5 - On-Time Delivery Rate: {on_time_rate:.1f}%")
print(f"         Late Delivery Rate    : {late_rate:.1f}%")

# Visualize as a donut chart for clean dashboard presentation
fig3 = go.Figure(data=[go.Pie(
    labels=['On Time', 'Late'],
    values=[on_time_rate, late_rate],
    hole=0.55,
    marker_colors=['#2563eb', '#ef4444'],
    textinfo='label+percent',
    textfont_size=14
)])
fig3.update_layout(
    title=dict(text='On-Time Delivery Rate', font_size=18),
    showlegend=False,
    annotations=[dict(
        text=f"{on_time_rate:.1f}%<br>On Time",
        x=0.5, y=0.5,
        font_size=16,
        showarrow=False
    )]
)
fig3.show()

# ────────────────────────────────────────────
# KPI 6: Average Delivery Delay
# ────────────────────────────────────────────
# Average number of days late across all late orders.
# We separate late and early orders to avoid the two effects
# cancelling each other out in the average.
avg_delay_late = delivery_df[delivery_df['is_late'] == 1]['delivery_delay_days'].mean()
avg_early = delivery_df[delivery_df['is_on_time'] == 1]['delivery_delay_days'].mean()

print(f"\nKPI 6 - Avg delay for late orders : {avg_delay_late:.1f} days late")
print(f"         Avg days early for on-time : {abs(avg_early):.1f} days early")

# ────────────────────────────────────────────
# KPI 7: Delivery Delay Distribution
# ────────────────────────────────────────────
# A histogram of delivery delay days shows us the shape of the
# delay problem — is it a few extreme outliers or a systemic issue?
# This helps operations teams decide whether to fix processes broadly
# or focus on specific edge cases.
fig4 = px.histogram(
    delivery_df,
    x='delivery_delay_days',
    nbins=80,
    title='Distribution of Delivery Delay (days)',
    labels={'delivery_delay_days': 'Delay in Days (negative = early, positive = late)'},
    color_discrete_sequence=['#2563eb']
)
fig4.add_vline(
    x=0,
    line_dash='dash',
    line_color='red',
    annotation_text='On-Time Threshold',
    annotation_position='top right'
)
fig4.update_layout(
    title_font_size=18,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey', title='Number of Orders')
)
fig4.show()

# ── Insight ──
print("\nInsight - Delivery Delay Distribution:")
pct_more_than_7 = (delivery_df['delivery_delay_days'] > 7).mean() * 100
print(f"  Orders more than 7 days late: {pct_more_than_7:.1f}%")
pct_early = (delivery_df['delivery_delay_days'] < 0).mean() * 100
print(f"  Orders delivered early: {pct_early:.1f}%")

# ────────────────────────────────────────────
# KPI 8: Delivery Performance by State
# ────────────────────────────────────────────
# Late delivery rates vary significantly by region due to differences
# in logistics infrastructure. This chart helps identify which states
# need the most operational attention.
state_delivery = (
    delivery_df
    .groupby('customer_state')
    .agg(
        total_orders=('order_id', 'count'),
        late_delivery_pct=('is_late', lambda x: x.mean() * 100),
        avg_delay_days=('delivery_delay_days', 'mean')
    )
    .reset_index()
    .sort_values('late_delivery_pct', ascending=False)
    .head(15)
)

fig5 = px.bar(
    state_delivery,
    x='customer_state',
    y='late_delivery_pct',
    title='Late Delivery Rate by Customer State (Top 15)',
    labels={
        'customer_state': 'State',
        'late_delivery_pct': 'Late Delivery Rate (%)'
    },
    color='late_delivery_pct',
    color_continuous_scale='Reds',
    text=state_delivery['late_delivery_pct'].round(1).astype(str) + '%'
)
fig5.update_layout(
    title_font_size=18,
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey')
)
fig5.update_traces(textposition='outside')
fig5.show()

# ── Insight ──
print("\nInsight - Delivery Performance by State:")
worst_state = state_delivery.iloc[0]
best_state = state_delivery.iloc[-1]
print(f"  Worst performing state: {worst_state['customer_state']} "
      f"({worst_state['late_delivery_pct']:.1f}% late)")
print(f"  Best in top 15       : {best_state['customer_state']} "
      f"({best_state['late_delivery_pct']:.1f}% late)")

print("\nOperations KPIs complete")

Orders with delivery data: 96,470

KPI 5 - On-Time Delivery Rate: 93.2%
         Late Delivery Rate    : 6.8%



KPI 6 - Avg delay for late orders : 10.6 days late
         Avg days early for on-time : 13.5 days early



Insight - Delivery Delay Distribution:
  Orders more than 7 days late: 3.0%
  Orders delivered early: 91.9%



Insight - Delivery Performance by State:
  Worst performing state: AL (21.4% late)
  Best in top 15       : RN (9.3% late)

Operations KPIs complete


In [33]:
# ── Seller & Customer KPIs ──
# These KPIs measure customer satisfaction and purchasing behavior.
# Review scores are a direct proxy for customer experience, while
# order patterns by time reveal demand seasonality — both critical
# for operational planning and forecasting.
# We define four KPIs here:
# 1. Review Score Distribution
# 2. Sentiment Breakdown
# 3. Monthly Order Volume Trend
# 4. Orders by Day of Week

# ────────────────────────────────────────────
# KPI 9: Review Score Distribution
# ────────────────────────────────────────────
# Understanding how customers rate their experience tells us whether
# satisfaction is improving or declining over time. A skew toward
# 1-star reviews is a leading indicator of churn.
review_dist = (
    delivered
    .dropna(subset=['avg_review_score'])
    .assign(review_score_rounded=lambda x: x['avg_review_score'].round())
    .groupby('review_score_rounded')['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'order_count', 'review_score_rounded': 'review_score'})
)

fig6 = px.bar(
    review_dist,
    x='review_score',
    y='order_count',
    title='Review Score Distribution',
    labels={'review_score': 'Review Score (Stars)', 'order_count': 'Number of Orders'},
    color='review_score',
    color_continuous_scale='Blues',
    text='order_count'
)
fig6.update_layout(
    title_font_size=18,
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(gridcolor='lightgrey', tickmode='linear'),
    yaxis=dict(gridcolor='lightgrey')
)
fig6.update_traces(textposition='outside')
fig6.show()

# ── Insight ──
avg_score = delivered['avg_review_score'].mean()
pct_5_star = (delivered['avg_review_score'].round() == 5).mean() * 100
print(f"\nInsight - Review Scores:")
print(f"  Average review score : {avg_score:.2f} / 5.0")
print(f"  5-star orders        : {pct_5_star:.1f}%")

# ────────────────────────────────────────────
# KPI 10: Sentiment Breakdown
# ────────────────────────────────────────────
# We bucket scores into positive, neutral, and negative sentiment
# for a higher-level view of customer satisfaction that is easier
# to communicate to non-technical stakeholders.

# Fix for newer Pandas versions where value_counts() column names differ
sentiment_counts = (
    delivered['review_sentiment']
    .value_counts()
    .reset_index()
)
sentiment_counts.columns = ['review_sentiment', 'count']

fig7 = px.pie(
    sentiment_counts,
    names='review_sentiment',
    values='count',
    title='Customer Sentiment Breakdown',
    color='review_sentiment',
    color_discrete_map={
        'positive': '#2563eb',
        'neutral': '#f59e0b',
        'negative': '#ef4444',
        'no review': '#9ca3af'
    }
)
fig7.update_layout(title_font_size=18)
fig7.update_traces(textinfo='label+percent', textfont_size=13)
fig7.show()

# ── Insight ──
print("\nInsight - Sentiment Breakdown:")
for _, row in sentiment_counts.iterrows():
    pct = row['count'] / sentiment_counts['count'].sum() * 100
    print(f"  {row['review_sentiment']:<12}: {row['count']:,} orders ({pct:.1f}%)")

# ────────────────────────────────────────────
# KPI 11: Monthly Order Volume Trend
# ────────────────────────────────────────────
# Revenue tells us how much money came in, but order volume tells us
# how many customers we served. Comparing both trends reveals whether
# revenue growth is driven by more orders or higher order values.
monthly_orders = (
    delivered
    .groupby(['purchase_year', 'purchase_month'])['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'order_count'})
)
monthly_orders['period'] = pd.to_datetime(
    monthly_orders['purchase_year'].astype(str) + '-' +
    monthly_orders['purchase_month'].astype(str).str.zfill(2)
)
monthly_orders = monthly_orders.sort_values('period')

# Remove last incomplete month
last_month = monthly_orders['period'].max()
monthly_orders = monthly_orders[monthly_orders['period'] < last_month]

fig8 = px.bar(
    monthly_orders,
    x='period',
    y='order_count',
    title='Monthly Order Volume Trend',
    labels={'period': 'Month', 'order_count': 'Number of Orders'},
    color_discrete_sequence=['#2563eb']
)
fig8.update_layout(
    title_font_size=18,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey')
)
fig8.show()

# ── Insight ──
peak_month_orders = monthly_orders.loc[monthly_orders['order_count'].idxmax()]
print(f"\nInsight - Order Volume:")
print(f"  Peak order month: {peak_month_orders['period'].strftime('%B %Y')} "
      f"({peak_month_orders['order_count']:,} orders)")
print(f"  Avg monthly orders: {monthly_orders['order_count'].mean():,.0f}")

# ────────────────────────────────────────────
# KPI 12: Orders by Day of Week
# ────────────────────────────────────────────
# Day of week patterns reveal when customers prefer to shop.
# This informs staffing, promotion scheduling, and inventory
# replenishment timing.
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_orders = (
    delivered
    .groupby('purchase_dayofweek')['order_id']
    .count()
    .reindex(day_order)
    .reset_index()
    .rename(columns={'order_id': 'order_count'})
)

fig9 = px.bar(
    dow_orders,
    x='purchase_dayofweek',
    y='order_count',
    title='Order Volume by Day of Week',
    labels={'purchase_dayofweek': 'Day of Week', 'order_count': 'Number of Orders'},
    color='order_count',
    color_continuous_scale='Blues'
)
fig9.update_layout(
    title_font_size=18,
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey')
)
fig9.show()

# ── Insight ──
busiest_day = dow_orders.loc[dow_orders['order_count'].idxmax(), 'purchase_dayofweek']
quietest_day = dow_orders.loc[dow_orders['order_count'].idxmin(), 'purchase_dayofweek']
print(f"\nInsight - Day of Week Patterns:")
print(f"  Busiest day : {busiest_day}")
print(f"  Quietest day: {quietest_day}")

print("\nSeller & Customer KPIs complete")


Insight - Review Scores:
  Average review score : 4.16 / 5.0
  5-star orders        : 58.8%



Insight - Sentiment Breakdown:
  positive    : 75,626 orders (78.4%)
  negative    : 12,291 orders (12.7%)
  neutral     : 7,915 orders (8.2%)
  no review   : 646 orders (0.7%)



Insight - Order Volume:
  Peak order month: November 2017 (7,289 orders)
  Avg monthly orders: 4,097



Insight - Day of Week Patterns:
  Busiest day : Monday
  Quietest day: Saturday

Seller & Customer KPIs complete


In [34]:
# ── KPI Summary Dashboard ──
# A consolidated single-view summary of all key metrics computed in
# this section. This is what you would present to a business stakeholder
# who wants a quick snapshot of the business without going through
# every individual chart. It also demonstrates the ability to translate
# complex data into a clean, readable executive summary — a core
# requirement of the job description.

# ── Compute all summary KPI values ──
total_orders = len(delivered)
total_revenue = delivered['total_order_value'].sum()
aov = delivered['total_order_value'].mean()
on_time_rate = delivery_df['is_on_time'].mean() * 100
avg_delay = delivery_df[delivery_df['is_late'] == 1]['delivery_delay_days'].mean()
avg_review = delivered['avg_review_score'].mean()
pct_positive = (sentiment_counts.loc[
    sentiment_counts['review_sentiment'] == 'positive', 'count'].values[0] /
    sentiment_counts['count'].sum() * 100)
top_category = (
    delivered.groupby('primary_category')['total_order_value']
    .sum().idxmax()
)

print("=" * 60)
print("        OLIST E-COMMERCE KPI SUMMARY DASHBOARD")
print("=" * 60)

print(f"""
FINANCE
-------
  Total Revenue          : R$ {total_revenue:>15,.2f}
  Total Delivered Orders :     {total_orders:>15,}
  Average Order Value    : R$ {aov:>15,.2f}
  Top Revenue Category   :     {top_category}

OPERATIONS
----------
  On-Time Delivery Rate  :     {on_time_rate:>14.1f}%
  Late Delivery Rate     :     {100 - on_time_rate:>14.1f}%
  Avg Delay (late orders):     {avg_delay:>13.1f} days

CUSTOMER SATISFACTION
---------------------
  Average Review Score   :     {avg_review:>14.2f} / 5.0
  Positive Sentiment     :     {pct_positive:>14.1f}%
  Busiest Shopping Day   :     {busiest_day}
""")
print("=" * 60)

# ── Visualize KPI summary as a metric card chart ──
# We use a Plotly figure with annotation boxes to simulate
# KPI metric cards — a common pattern in BI dashboards like
# Power BI and Tableau that stakeholders are familiar with.

fig10 = go.Figure()

kpis = [
    ("Total Revenue", f"R$ {total_revenue/1e6:.2f}M", "#2563eb"),
    ("Total Orders", f"{total_orders:,}", "#2563eb"),
    ("Avg Order Value", f"R$ {aov:.2f}", "#2563eb"),
    ("On-Time Delivery", f"{on_time_rate:.1f}%", "#16a34a" if on_time_rate >= 80 else "#ef4444"),
    ("Avg Delay (late)", f"{avg_delay:.1f} days", "#ef4444"),
    ("Avg Review Score", f"{avg_review:.2f} / 5.0", "#16a34a" if avg_review >= 4 else "#f59e0b"),
    ("Positive Sentiment", f"{pct_positive:.1f}%", "#16a34a"),
    ("Top Category", top_category[:20], "#2563eb"),
]

# Position cards in a 4x2 grid
cols = 4
for i, (label, value, color) in enumerate(kpis):
    row = i // cols
    col = i % cols
    x = col / cols + 0.01
    y = 1 - row * 0.5

    # Card background
    fig10.add_shape(
        type='rect',
        x0=x, x1=x + 0.22,
        y0=y - 0.38, y1=y + 0.08,
        line=dict(color=color, width=2),
        fillcolor='white',
        xref='paper', yref='paper'
    )
    # KPI value
    fig10.add_annotation(
        x=x + 0.11, y=y - 0.08,
        text=f"<b>{value}</b>",
        showarrow=False,
        font=dict(size=16, color=color),
        xref='paper', yref='paper',
        align='center'
    )
    # KPI label
    fig10.add_annotation(
        x=x + 0.11, y=y - 0.26,
        text=label,
        showarrow=False,
        font=dict(size=11, color='#6b7280'),
        xref='paper', yref='paper',
        align='center'
    )

fig10.update_layout(
    title=dict(
        text='Olist E-Commerce KPI Dashboard',
        font=dict(size=20)
    ),
    height=400,
    plot_bgcolor='#f9fafb',
    paper_bgcolor='#f9fafb',
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    margin=dict(l=20, r=20, t=60, b=20)
)
fig10.show()

print("\nKPI Summary Dashboard complete")
print("Section 3 complete — ready for anomaly detection")

        OLIST E-COMMERCE KPI SUMMARY DASHBOARD

FINANCE
-------
  Total Revenue          : R$   15,419,773.75
  Total Delivered Orders :              96,478
  Average Order Value    : R$          159.83
  Top Revenue Category   :     health_beauty

OPERATIONS
----------
  On-Time Delivery Rate  :               93.2%
  Late Delivery Rate     :                6.8%
  Avg Delay (late orders):              10.6 days

CUSTOMER SATISFACTION
---------------------
  Average Review Score   :               4.16 / 5.0
  Positive Sentiment     :               78.4%
  Busiest Shopping Day   :     Monday




KPI Summary Dashboard complete
Section 3 complete — ready for anomaly detection


Section 4 — Anomaly Detection

This section has the following chunks:

1. Daily Revenue Anomaly Detection — flag statistically abnormal days in revenue using Z-score and IQR methods
2. Delivery Anomaly Detection — flag days with abnormally high late delivery rates
3. Anomaly Summary Report — consolidate all flagged anomalies into a single readable output

In [35]:
# ── SECTION 4: Anomaly Detection ──

# ── Daily Revenue Anomaly Detection ──
# Anomaly detection on revenue helps the business identify days where
# something unusual happened — a flash sale, a system outage, a data
# pipeline error, or an external event. Catching these early prevents
# bad data from corrupting reports and helps operations teams respond
# quickly to unexpected drops or spikes.
#
# We use two complementary statistical methods:
# 1. Z-Score: flags values that are more than 3 standard deviations
#    from the mean. Best for normally distributed data.
# 2. IQR (Interquartile Range): flags values below Q1 - 1.5*IQR or
#    above Q3 + 1.5*IQR. More robust to skewed distributions.
# Flagging with both methods makes our anomaly detection more reliable.

# ── Aggregate daily revenue ──
# We work at the daily level because that is the granularity at which
# operational teams can actually investigate and act on anomalies.
daily_revenue = (
    delivered
    .dropna(subset=['purchase_date', 'total_order_value'])
    .groupby('purchase_date')['total_order_value']
    .sum()
    .reset_index()
    .rename(columns={'total_order_value': 'daily_revenue'})
    .sort_values('purchase_date')
)
daily_revenue['purchase_date'] = pd.to_datetime(daily_revenue['purchase_date'])

print(f"Daily revenue records: {len(daily_revenue):,} days")
print(f"Date range: {daily_revenue['purchase_date'].min().date()} "
      f"to {daily_revenue['purchase_date'].max().date()}")

# ── Z-Score method ──
# Z-score measures how many standard deviations a value is from the mean.
# A threshold of 3 is the standard choice — it catches clear outliers
# while avoiding too many false positives on natural variation.
mean_rev = daily_revenue['daily_revenue'].mean()
std_rev = daily_revenue['daily_revenue'].std()
daily_revenue['z_score'] = (
    (daily_revenue['daily_revenue'] - mean_rev) / std_rev
).round(3)
daily_revenue['z_anomaly'] = daily_revenue['z_score'].abs() > 3

# ── IQR method ──
# IQR is more robust than Z-score when data is skewed (which revenue
# data often is — a few very high days can inflate the mean and std).
Q1 = daily_revenue['daily_revenue'].quantile(0.25)
Q3 = daily_revenue['daily_revenue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

daily_revenue['iqr_anomaly'] = (
    (daily_revenue['daily_revenue'] < lower_bound) |
    (daily_revenue['daily_revenue'] > upper_bound)
)

# ── Combined anomaly flag ──
# We flag a day as a confirmed anomaly only if BOTH methods agree.
# This reduces false positives — a day flagged by only one method
# may just be natural variation at the edge of the distribution.
daily_revenue['is_revenue_anomaly'] = (
    daily_revenue['z_anomaly'] & daily_revenue['iqr_anomaly']
)

n_anomalies = daily_revenue['is_revenue_anomaly'].sum()
print(f"\nRevenue anomalies detected: {n_anomalies} days "
      f"({n_anomalies/len(daily_revenue)*100:.1f}% of all days)")

# ── Visualize daily revenue with anomalies highlighted ──
fig11 = go.Figure()

# Normal days
normal = daily_revenue[~daily_revenue['is_revenue_anomaly']]
fig11.add_trace(go.Scatter(
    x=normal['purchase_date'],
    y=normal['daily_revenue'],
    mode='lines',
    name='Normal',
    line=dict(color='#2563eb', width=1.5)
))

# Anomalous days
anomalies = daily_revenue[daily_revenue['is_revenue_anomaly']]
fig11.add_trace(go.Scatter(
    x=anomalies['purchase_date'],
    y=anomalies['daily_revenue'],
    mode='markers',
    name='Anomaly',
    marker=dict(color='#ef4444', size=10, symbol='circle')
))

# Upper and lower IQR bounds as reference lines
fig11.add_hline(
    y=upper_bound,
    line_dash='dash',
    line_color='orange',
    annotation_text=f'IQR Upper Bound (R$ {upper_bound:,.0f})',
    annotation_position='top left'
)
fig11.add_hline(
    y=lower_bound,
    line_dash='dash',
    line_color='orange',
    annotation_text=f'IQR Lower Bound (R$ {lower_bound:,.0f})',
    annotation_position='bottom left'
)

fig11.update_layout(
    title='Daily Revenue with Anomalies Highlighted',
    title_font_size=18,
    xaxis_title='Date',
    yaxis_title='Daily Revenue (R$)',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig11.show()

# ── Print anomalous days ──
print("\nFlagged Revenue Anomaly Days:")
print(anomalies[['purchase_date', 'daily_revenue', 'z_score']]
      .sort_values('daily_revenue', ascending=False)
      .to_string(index=False))

Daily revenue records: 612 days
Date range: 2016-09-15 to 2018-08-29

Revenue anomalies detected: 2 days (0.3% of all days)



Flagged Revenue Anomaly Days:
purchase_date  daily_revenue  z_score
   2017-11-24      175178.46    10.44
   2017-11-25       70683.26     3.17


In [36]:
# ── Delivery Anomaly Detection ──
# Just as we flagged abnormal revenue days, we now flag days where
# the late delivery rate was statistically abnormal. A sudden spike
# in late deliveries on a specific day can indicate a carrier issue,
# a warehouse problem, or a weather event affecting logistics.
# Catching these patterns early allows operations teams to proactively
# communicate with affected customers and investigate root causes.

# ── Aggregate daily delivery performance ──
# We compute the late delivery rate per day — the percentage of orders
# that were delivered late out of all orders delivered that day.
daily_delivery = (
    delivery_df
    .dropna(subset=['purchase_date', 'is_late'])
    .groupby('purchase_date')
    .agg(
        total_orders=('order_id', 'count'),
        late_orders=('is_late', 'sum'),
        avg_delay_days=('delivery_delay_days', 'mean')
    )
    .reset_index()
)
daily_delivery['purchase_date'] = pd.to_datetime(daily_delivery['purchase_date'])
daily_delivery['late_rate'] = (
    daily_delivery['late_orders'] / daily_delivery['total_orders'] * 100
).round(2)

# ── Filter to days with enough orders to be statistically meaningful ──
# A day with only 1-2 orders where one is late would show a 50-100%
# late rate, which is misleading. We require at least 10 orders per
# day before flagging anomalies.
daily_delivery = daily_delivery[daily_delivery['total_orders'] >= 10].copy()
print(f"Days with sufficient order volume (>=10): {len(daily_delivery):,}")

# ── Z-Score anomaly detection on late delivery rate ──
mean_late = daily_delivery['late_rate'].mean()
std_late = daily_delivery['late_rate'].std()
daily_delivery['late_z_score'] = (
    (daily_delivery['late_rate'] - mean_late) / std_late
).round(3)
daily_delivery['z_anomaly'] = daily_delivery['late_z_score'].abs() > 3

# ── IQR anomaly detection on late delivery rate ──
Q1_late = daily_delivery['late_rate'].quantile(0.25)
Q3_late = daily_delivery['late_rate'].quantile(0.75)
IQR_late = Q3_late - Q1_late
upper_late = Q3_late + 1.5 * IQR_late
lower_late = Q1_late - 1.5 * IQR_late

daily_delivery['iqr_anomaly'] = (
    (daily_delivery['late_rate'] > upper_late) |
    (daily_delivery['late_rate'] < lower_late)
)

# ── Combined anomaly flag ──
# Again we require both methods to agree before flagging a day.
daily_delivery['is_delivery_anomaly'] = (
    daily_delivery['z_anomaly'] & daily_delivery['iqr_anomaly']
)

n_delivery_anomalies = daily_delivery['is_delivery_anomaly'].sum()
print(f"\nDelivery anomalies detected: {n_delivery_anomalies} days "
      f"({n_delivery_anomalies/len(daily_delivery)*100:.1f}% of all valid days)")

# ── Visualize daily late delivery rate with anomalies highlighted ──
fig12 = go.Figure()

# Normal days
normal_del = daily_delivery[~daily_delivery['is_delivery_anomaly']]
fig12.add_trace(go.Scatter(
    x=normal_del['purchase_date'],
    y=normal_del['late_rate'],
    mode='lines',
    name='Normal',
    line=dict(color='#2563eb', width=1.5)
))

# Anomalous days
anomalies_del = daily_delivery[daily_delivery['is_delivery_anomaly']]
fig12.add_trace(go.Scatter(
    x=anomalies_del['purchase_date'],
    y=anomalies_del['late_rate'],
    mode='markers',
    name='Anomaly',
    marker=dict(color='#ef4444', size=10, symbol='circle')
))

# IQR bounds
fig12.add_hline(
    y=upper_late,
    line_dash='dash',
    line_color='orange',
    annotation_text=f'IQR Upper Bound ({upper_late:.1f}%)',
    annotation_position='top left'
)
fig12.add_hline(
    y=mean_late,
    line_dash='dot',
    line_color='green',
    annotation_text=f'Mean Late Rate ({mean_late:.1f}%)',
    annotation_position='bottom right'
)

fig12.update_layout(
    title='Daily Late Delivery Rate with Anomalies Highlighted',
    title_font_size=18,
    xaxis_title='Date',
    yaxis_title='Late Delivery Rate (%)',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig12.show()

# ── Print flagged delivery anomaly days ──
if n_delivery_anomalies > 0:
    print("\nFlagged Delivery Anomaly Days:")
    print(anomalies_del[['purchase_date', 'total_orders', 'late_orders',
                          'late_rate', 'late_z_score']]
          .sort_values('late_rate', ascending=False)
          .to_string(index=False))
else:
    print("\nNo confirmed delivery anomalies detected with combined method.")
    print(f"Days exceeding IQR upper bound ({upper_late:.1f}%): "
          f"{daily_delivery['iqr_anomaly'].sum()}")

Days with sufficient order volume (>=10): 604

Delivery anomalies detected: 17 days (2.8% of all valid days)



Flagged Delivery Anomaly Days:
purchase_date  total_orders  late_orders  late_rate  late_z_score
   2018-03-02           262           79      30.15          4.57
   2018-03-12           228           65      28.51          4.27
   2018-03-06           268           74      27.61          4.10
   2018-02-26           290           79      27.24          4.03
   2018-03-03           205           55      26.83          3.96
   2018-03-04           228           61      26.75          3.94
   2018-03-05           252           67      26.59          3.91
   2018-03-01           268           71      26.49          3.89
   2018-03-07           254           66      25.98          3.80
   2018-03-11           215           55      25.58          3.72
   2018-02-24           188           47      25.00          3.62
   2018-02-27           293           73      24.91          3.60
   2018-03-10           185           43      23.24          3.29
   2018-03-14           195           45    

In [37]:
# ── Anomaly Summary Report ──
# We consolidate all flagged anomalies from both revenue and delivery
# detection into a single structured report. In a production pipeline
# this report would be automatically emailed to operations and finance
# teams, or surfaced in a monitoring dashboard like Kibana or Power BI.
# Here we simulate that output as a clean printed report and a combined
# visualization — demonstrating the ability to translate analytical
# findings into actionable business communication.

print("=" * 60)
print("         ANOMALY DETECTION SUMMARY REPORT")
print("=" * 60)

# ── Revenue Anomaly Summary ──
print("\nREVENUE ANOMALIES")
print("-" * 60)
print(f"  Detection methods    : Z-Score (threshold=3) + IQR (1.5x)")
print(f"  Days analyzed        : {len(daily_revenue):,}")
print(f"  Anomalies detected   : {daily_revenue['is_revenue_anomaly'].sum()}")
print(f"  Normal revenue range : R$ {lower_bound:,.2f} — R$ {upper_bound:,.2f}")
print(f"  Mean daily revenue   : R$ {mean_rev:,.2f}")
print(f"  Std dev              : R$ {std_rev:,.2f}")

if len(anomalies) > 0:
    print(f"\n  Flagged Days:")
    for _, row in anomalies.sort_values('daily_revenue', ascending=False).iterrows():
        direction = "SPIKE" if row['daily_revenue'] > upper_bound else "DROP"
        print(f"    {str(row['purchase_date'].date()):<15} "
              f"R$ {row['daily_revenue']:>10,.2f}   "
              f"Z={row['z_score']:>6.2f}   [{direction}]")

# ── Delivery Anomaly Summary ──
print("\nDELIVERY ANOMALIES")
print("-" * 60)
print(f"  Detection methods    : Z-Score (threshold=3) + IQR (1.5x)")
print(f"  Days analyzed        : {len(daily_delivery):,}")
print(f"  Anomalies detected   : {daily_delivery['is_delivery_anomaly'].sum()}")
print(f"  Normal late rate range: {lower_late:.1f}% — {upper_late:.1f}%")
print(f"  Mean late rate       : {mean_late:.1f}%")
print(f"  Std dev              : {std_late:.1f}%")

if n_delivery_anomalies > 0:
    print(f"\n  Flagged Days:")
    for _, row in anomalies_del.sort_values('late_rate', ascending=False).iterrows():
        print(f"    {str(row['purchase_date'].date()):<15} "
              f"Late Rate: {row['late_rate']:>6.1f}%   "
              f"Orders: {int(row['total_orders']):>5}   "
              f"Z={row['late_z_score']:>6.2f}")
else:
    print("\n  No confirmed delivery anomalies detected.")

print("\n" + "=" * 60)

# ── Combined anomaly visualization ──
# We plot both revenue and delivery anomaly counts per month
# to see if anomalies cluster around specific time periods —
# which could indicate seasonal issues or systemic pipeline problems.

daily_revenue['month'] = daily_revenue['purchase_date'].dt.to_period('M')
daily_delivery['month'] = daily_delivery['purchase_date'].dt.to_period('M')

monthly_rev_anomalies = (
    daily_revenue
    .groupby('month')['is_revenue_anomaly']
    .sum()
    .reset_index()
    .rename(columns={'is_revenue_anomaly': 'revenue_anomalies'})
)
monthly_del_anomalies = (
    daily_delivery
    .groupby('month')['is_delivery_anomaly']
    .sum()
    .reset_index()
    .rename(columns={'is_delivery_anomaly': 'delivery_anomalies'})
)

# Convert period to string for Plotly compatibility
monthly_rev_anomalies['month'] = monthly_rev_anomalies['month'].astype(str)
monthly_del_anomalies['month'] = monthly_del_anomalies['month'].astype(str)

# Merge both anomaly series on month
anomaly_summary = monthly_rev_anomalies.merge(
    monthly_del_anomalies, on='month', how='outer'
).fillna(0).sort_values('month')

fig13 = go.Figure()

fig13.add_trace(go.Bar(
    x=anomaly_summary['month'],
    y=anomaly_summary['revenue_anomalies'],
    name='Revenue Anomalies',
    marker_color='#2563eb'
))

fig13.add_trace(go.Bar(
    x=anomaly_summary['month'],
    y=anomaly_summary['delivery_anomalies'],
    name='Delivery Anomalies',
    marker_color='#ef4444'
))

fig13.update_layout(
    title='Monthly Anomaly Count — Revenue vs Delivery',
    title_font_size=18,
    xaxis_title='Month',
    yaxis_title='Number of Anomalous Days',
    barmode='group',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey', tickangle=45),
    yaxis=dict(gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig13.show()

print("\nAnomaly Detection complete")
print("Section 4 complete — ready for forecasting")

         ANOMALY DETECTION SUMMARY REPORT

REVENUE ANOMALIES
------------------------------------------------------------
  Detection methods    : Z-Score (threshold=3) + IQR (1.5x)
  Days analyzed        : 612
  Anomalies detected   : 2
  Normal revenue range : R$ -13,396.02 — R$ 61,826.05
  Mean daily revenue   : R$ 25,195.71
  Std dev              : R$ 14,370.47

  Flagged Days:
    2017-11-24      R$ 175,178.46   Z= 10.44   [SPIKE]
    2017-11-25      R$  70,683.26   Z=  3.17   [SPIKE]

DELIVERY ANOMALIES
------------------------------------------------------------
  Detection methods    : Z-Score (threshold=3) + IQR (1.5x)
  Days analyzed        : 604
  Anomalies detected   : 17
  Normal late rate range: -5.2% — 14.1%
  Mean late rate       : 5.5%
  Std dev              : 5.4%

  Flagged Days:
    2018-03-02      Late Rate:   30.1%   Orders:   262   Z=  4.57
    2018-03-12      Late Rate:   28.5%   Orders:   228   Z=  4.27
    2018-03-06      Late Rate:   27.6%   Orders:   268   Z


Anomaly Detection complete
Section 4 complete — ready for forecasting


Section 5 — Forecasting

This section has the following chunks:

1. Install & Prepare Data for Prophet — install Facebook Prophet and prepare the daily revenue time series
2. Train the Forecast Model — fit Prophet on historical data and generate future predictions
3. Visualize & Interpret the Forecast — plot actual vs predicted revenue and extract business insights

In [38]:
# ── SECTION 5: Forecasting ──

# ── Install & Prepare Data for Prophet ──
# Facebook Prophet is a time-series forecasting library developed by
# Meta's data science team. It is widely used in industry for business
# forecasting because it handles seasonality, holidays, and missing data
# well without requiring heavy statistical expertise to configure.
# It is particularly well suited for daily business metrics like revenue
# where weekly and yearly seasonality patterns are common.

!pip install prophet --quiet

from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
import plotly.graph_objects as go

print("Prophet installed and imported successfully")

# ── Prepare the time series data ──
# Prophet requires a very specific input format:
# a DataFrame with exactly two columns:
#   'ds' — the date column (datetime)
#   'y'  — the value to forecast (numeric)
# Any other column names will cause an error.

# We use daily revenue from all delivered orders as our target series.
# We reuse the daily_revenue DataFrame from Section 4 which already
# has clean dates and aggregated revenue values.
prophet_df = daily_revenue[['purchase_date', 'daily_revenue']].copy()
prophet_df = prophet_df.rename(columns={
    'purchase_date': 'ds',
    'daily_revenue': 'y'
})

# Sort by date to ensure the time series is in chronological order
prophet_df = prophet_df.sort_values('ds').reset_index(drop=True)

# ── Remove anomalous days from training data ──
# Anomalous days identified in Section 4 can distort the model's
# understanding of normal patterns. We cap them at the IQR upper
# bound rather than removing them entirely — this preserves the
# data point in the series while preventing extreme values from
# pulling the trend line in the wrong direction.
prophet_df['y'] = prophet_df['y'].clip(upper=upper_bound)

print(f"\nForecast training data:")
print(f"  Date range : {prophet_df['ds'].min().date()} to {prophet_df['ds'].max().date()}")
print(f"  Total days : {len(prophet_df):,}")
print(f"  Mean daily revenue : R$ {prophet_df['y'].mean():,.2f}")
print(f"  Max daily revenue  : R$ {prophet_df['y'].max():,.2f}")
print(f"  Min daily revenue  : R$ {prophet_df['y'].min():,.2f}")

print("\nSample of Prophet input data:")
display(prophet_df.head(5))

Prophet installed and imported successfully

Forecast training data:
  Date range : 2016-09-15 to 2018-08-29
  Total days : 612
  Mean daily revenue : R$ 24,991.06
  Max daily revenue  : R$ 61,826.05
  Min daily revenue  : R$ 19.62

Sample of Prophet input data:


,ds,y
0,2016-09-15,143.46
1,2016-10-03,559.53
2,2016-10-04,9821.42
3,2016-10-05,7209.50
4,2016-10-06,6798.90


In [39]:
# ── Train the Forecast Model ──
# We initialize and fit a Prophet model on our historical daily revenue.
# Prophet decomposes the time series into three components:
#   1. Trend      — the overall long-term direction of revenue
#   2. Seasonality — repeating patterns (weekly, yearly)
#   3. Residual   — what is left after trend and seasonality are removed
#
# We configure the model with settings appropriate for e-commerce
# revenue data where both weekly shopping patterns and yearly
# seasonal peaks (e.g. holidays, Black Friday) are expected.

# ── Initialize Prophet model ──
model = Prophet(
    # yearly_seasonality captures annual patterns like holiday season
    # spikes and post-holiday slowdowns common in e-commerce
    yearly_seasonality=True,

    # weekly_seasonality captures day-of-week patterns — we already
    # saw in Section 3 that order volume varies significantly by day
    weekly_seasonality=True,

    # We disable daily seasonality because our data is already
    # aggregated at the daily level — there is no sub-daily pattern
    daily_seasonality=False,

    # changepoint_prior_scale controls how flexible the trend is.
    # A higher value allows the trend to change more sharply at
    # changepoints. 0.1 is a moderate value suitable for business
    # revenue that can shift but should not overfit to noise.
    changepoint_prior_scale=0.1,

    # seasonality_prior_scale controls how strongly seasonality
    # patterns are fitted. 10 is the Prophet default and works
    # well for e-commerce data with clear seasonal patterns.
    seasonality_prior_scale=10,

    # interval_width sets the width of the uncertainty interval
    # shown in the forecast. 0.95 means we are 95% confident the
    # true value will fall within the shaded band.
    interval_width=0.95
)

# ── Fit the model on historical data ──
# This is where Prophet learns the trend and seasonality patterns
# from our daily revenue time series.
print("Training Prophet model...")
model.fit(prophet_df)
print("Model training complete")

# ── Create future dataframe for predictions ──
# We ask Prophet to forecast 90 days beyond the last date in our
# training data. 90 days is a practical planning horizon for
# operations and finance teams — long enough to be useful for
# quarterly planning, short enough to remain reliable.
future = model.make_future_dataframe(periods=90, freq='D')

print(f"\nForecast horizon: 90 days")
print(f"  Training ends  : {prophet_df['ds'].max().date()}")
print(f"  Forecast ends  : {future['ds'].max().date()}")

# ── Generate predictions ──
# The forecast DataFrame contains:
#   'yhat'       — the predicted value
#   'yhat_lower' — lower bound of the 95% confidence interval
#   'yhat_upper' — upper bound of the 95% confidence interval
#   'trend'      — the trend component alone
forecast = model.predict(future)

print(f"\nForecast generated for {len(future):,} days total "
      f"({len(prophet_df):,} historical + 90 future)")

# ── Preview forecast output ──
forecast_preview = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend']].tail(10)
forecast_preview.columns = ['Date', 'Predicted Revenue', 'Lower Bound', 'Upper Bound', 'Trend']
for col in ['Predicted Revenue', 'Lower Bound', 'Upper Bound', 'Trend']:
    forecast_preview[col] = forecast_preview[col].apply(lambda x: f"R$ {x:,.2f}")

print("\nSample of 90-day forecast (last 10 days):")
display(forecast_preview)

Training Prophet model...
Model training complete

Forecast horizon: 90 days
  Training ends  : 2018-08-29
  Forecast ends  : 2018-11-27

Forecast generated for 702 days total (612 historical + 90 future)

Sample of 90-day forecast (last 10 days):


,Date,Predicted Revenue,Lower Bound,Upper Bound,Trend
692,2018-11-18,"R$ 37,505.74","R$ 25,423.76","R$ 49,251.90","R$ 34,242.71"
693,2018-11-19,"R$ 47,168.02","R$ 34,315.22","R$ 59,780.65","R$ 34,238.91"
694,2018-11-20,"R$ 47,930.80","R$ 35,675.86","R$ 59,790.39","R$ 34,235.10"
695,2018-11-21,"R$ 48,276.84","R$ 35,129.48","R$ 60,528.47","R$ 34,231.30"
696,2018-11-22,"R$ 48,470.58","R$ 35,859.98","R$ 60,904.31","R$ 34,227.50"
697,2018-11-23,"R$ 47,568.99","R$ 35,295.88","R$ 59,832.85","R$ 34,223.69"
698,2018-11-24,"R$ 43,605.65","R$ 31,296.75","R$ 55,637.56","R$ 34,219.89"
699,2018-11-25,"R$ 45,574.81","R$ 33,092.63","R$ 58,113.66","R$ 34,216.08"
700,2018-11-26,"R$ 54,347.77","R$ 41,735.95","R$ 66,999.19","R$ 34,212.28"
701,2018-11-27,"R$ 54,082.42","R$ 42,044.19","R$ 66,854.52","R$ 34,208.47"


In [40]:
# ── Visualize & Interpret the Forecast ──
# The final step is to present the forecast in a way that is clear
# and actionable for both technical and non-technical stakeholders.
# We produce three outputs:
#   1. Actual vs Predicted revenue chart with confidence intervals
#   2. Forecast components breakdown (trend + seasonality)
#   3. A written business interpretation of the forecast findings

# ── Chart 1: Actual vs Predicted Revenue ──
# We overlay the historical actuals with the model's fitted values
# and the 90-day forward forecast. The shaded band represents the
# 95% confidence interval — wider bands mean more uncertainty.

fig14 = go.Figure()

# Historical actual revenue
fig14.add_trace(go.Scatter(
    x=prophet_df['ds'],
    y=prophet_df['y'],
    mode='lines',
    name='Actual Revenue',
    line=dict(color='#2563eb', width=1.5)
))

# Model fitted values on historical period
historical_forecast = forecast[forecast['ds'] <= prophet_df['ds'].max()]
fig14.add_trace(go.Scatter(
    x=historical_forecast['ds'],
    y=historical_forecast['yhat'],
    mode='lines',
    name='Model Fit',
    line=dict(color='#16a34a', width=1.5, dash='dot')
))

# 90-day forward forecast
future_forecast = forecast[forecast['ds'] > prophet_df['ds'].max()]
fig14.add_trace(go.Scatter(
    x=future_forecast['ds'],
    y=future_forecast['yhat'],
    mode='lines',
    name='Forecast (90 days)',
    line=dict(color='#f59e0b', width=2)
))

# Confidence interval shading for the forecast period
fig14.add_trace(go.Scatter(
    x=pd.concat([future_forecast['ds'], future_forecast['ds'][::-1]]),
    y=pd.concat([future_forecast['yhat_upper'], future_forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(245, 158, 11, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% Confidence Interval'
))

# ── Draw forecast start line as a vertical scatter trace ──
# We avoid add_vline entirely due to a Plotly/Pandas timestamp
# compatibility issue in this environment. A vertical scatter
# trace achieves the same visual result reliably.
forecast_start = prophet_df['ds'].max()
fig14.add_trace(go.Scatter(
    x=[forecast_start, forecast_start],
    y=[0, prophet_df['y'].max() * 1.1],
    mode='lines',
    name='Forecast Start',
    line=dict(color='grey', dash='dash', width=1.5),
    showlegend=True
))

fig14.update_layout(
    title='Daily Revenue — Actual vs Forecast (90 Days)',
    title_font_size=18,
    xaxis_title='Date',
    yaxis_title='Daily Revenue (R$)',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    hovermode='x unified'
)
fig14.show()

# ── Chart 2: Forecast Components ──
# Prophet decomposes the forecast into trend and seasonality components.
# This is extremely valuable for business communication — it lets us
# explain not just what will happen but why, in terms stakeholders
# can understand (e.g. "revenue dips every weekend" or "we see a
# strong growth trend through Q3").
fig15 = plot_components_plotly(model, forecast)
fig15.update_layout(
    title='Forecast Components — Trend & Seasonality Breakdown',
    title_font_size=18
)
fig15.show()

# ── Chart 3: 90-Day Forecast Summary by Month ──
# We aggregate the daily forecast into monthly totals for a cleaner
# executive-level view. Daily forecasts are noisy — monthly rollups
# are what finance and ops teams actually plan against.
future_forecast_copy = future_forecast.copy()
future_forecast_copy['month'] = future_forecast_copy['ds'].dt.to_period('M').astype(str)

monthly_forecast = (
    future_forecast_copy
    .groupby('month')
    .agg(
        forecasted_revenue=('yhat', 'sum'),
        lower_bound=('yhat_lower', 'sum'),
        upper_bound=('yhat_upper', 'sum')
    )
    .reset_index()
)

fig16 = go.Figure()
fig16.add_trace(go.Bar(
    x=monthly_forecast['month'],
    y=monthly_forecast['forecasted_revenue'],
    name='Forecasted Revenue',
    marker_color='#f59e0b',
    text=monthly_forecast['forecasted_revenue'].apply(lambda x: f"R$ {x/1000:.0f}K"),
    textposition='outside'
))
fig16.add_trace(go.Scatter(
    x=monthly_forecast['month'],
    y=monthly_forecast['upper_bound'],
    mode='lines',
    name='Upper Bound',
    line=dict(color='lightgrey', dash='dash')
))
fig16.add_trace(go.Scatter(
    x=monthly_forecast['month'],
    y=monthly_forecast['lower_bound'],
    mode='lines',
    name='Lower Bound',
    line=dict(color='lightgrey', dash='dash'),
    fill='tonexty',
    fillcolor='rgba(200,200,200,0.2)'
))
fig16.update_layout(
    title='90-Day Revenue Forecast — Monthly Rollup',
    title_font_size=18,
    xaxis_title='Month',
    yaxis_title='Forecasted Revenue (R$)',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgrey'),
    yaxis=dict(gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig16.show()

# ── Business Interpretation ──
avg_forecasted = future_forecast['yhat'].mean()
total_forecasted = future_forecast['yhat'].sum()
peak_forecast_day = future_forecast.loc[future_forecast['yhat'].idxmax()]
lowest_forecast_day = future_forecast.loc[future_forecast['yhat'].idxmin()]

print("=" * 60)
print("         FORECAST BUSINESS INTERPRETATION")
print("=" * 60)
print(f"""
FORECAST PERIOD : Next 90 days from {prophet_df['ds'].max().date()}

REVENUE OUTLOOK
---------------
  Total forecasted revenue  : R$ {total_forecasted:,.2f}
  Average daily revenue     : R$ {avg_forecasted:,.2f}
  Historical avg daily rev  : R$ {prophet_df['y'].mean():,.2f}
  Expected change           : {((avg_forecasted - prophet_df['y'].mean()) / prophet_df['y'].mean() * 100):+.1f}%

PEAK & TROUGH
-------------
  Highest forecast day : {peak_forecast_day['ds'].date()}
                         (R$ {peak_forecast_day['yhat']:,.2f})
  Lowest forecast day  : {lowest_forecast_day['ds'].date()}
                         (R$ {lowest_forecast_day['yhat']:,.2f})

MONTHLY FORECAST BREAKDOWN
---------------------------""")

for _, row in monthly_forecast.iterrows():
    print(f"  {row['month']} : R$ {row['forecasted_revenue']:>12,.2f} "
          f"  (Range: R$ {row['lower_bound']:,.0f} — R$ {row['upper_bound']:,.0f})")

print(f"""
KEY OBSERVATIONS
----------------
  - The trend component indicates whether revenue is growing or
    declining independent of seasonal effects.
  - Weekly seasonality confirms which days of the week drive the
    most orders — use this to time promotions and staffing.
  - The 95% confidence interval widens further into the future,
    reflecting increasing uncertainty — typical for any forecast.
  - These projections should be reviewed monthly and the model
    retrained as new data becomes available.
""")
print("=" * 60)
print("\nSection 5 complete — Project complete")
print("\nAll 5 sections finished:")
print("  Section 1 : Data Ingestion & Validation")
print("  Section 2 : Data Transformation & Feature Engineering")
print("  Section 3 : KPI Framework & EDA")
print("  Section 4 : Anomaly Detection")
print("  Section 5 : Forecasting")

         FORECAST BUSINESS INTERPRETATION

FORECAST PERIOD : Next 90 days from 2018-08-29

REVENUE OUTLOOK
---------------
  Total forecasted revenue  : R$ 3,064,488.11
  Average daily revenue     : R$ 34,049.87
  Historical avg daily rev  : R$ 24,991.06
  Expected change           : +36.2%

PEAK & TROUGH
-------------
  Highest forecast day : 2018-11-26 
                         (R$ 54,347.77)
  Lowest forecast day  : 2018-11-03 
                         (R$ 21,944.51)

MONTHLY FORECAST BREAKDOWN
---------------------------
  2018-08 : R$    57,290.86   (Range: R$ 33,847 — R$ 81,928)
  2018-09 : R$   968,690.93   (Range: R$ 600,132 — R$ 1,342,447)
  2018-10 : R$ 1,030,628.15   (Range: R$ 647,139 — R$ 1,412,418)
  2018-11 : R$ 1,007,878.17   (Range: R$ 670,514 — R$ 1,340,915)

KEY OBSERVATIONS
----------------
  - The trend component indicates whether revenue is growing or 
    declining independent of seasonal effects.
  - Weekly seasonality confirms which days of the week drive the 
